# ECOSETU — YOLOv8 V2 Material Detection Training Experiment

> **SIH Problem Statement 26229 — Kabadiwala Connect**  
> **Experiment:** YOLO V2 Controlled Fine-Tuning (`material-detection-v0.2.0`)  
> **Baseline V1 Reference:** `material-detection-v0.1.0` (Preserved Untouched)  
> **Target Dataset:** `material-detection-v0.1.0` (142 verified authentic images: 99 train, 28 val, 15 test)  
> **Key Adjustments:** Conservative `lr0 = 0.001`, `lrf = 0.01`, `patience = 30`, `seed = 42` to improve background calibration.  
> **Evaluation Policy:** Threshold selection strictly on validation data; held-out test evaluated once at selected threshold.  
> **Production Status:** `BASELINE V2 — EVALUATION REQUIRED`.

In [ ]:
# 1. Environment Setup & Dependency Installation
!pip install -q --upgrade pip
!pip install -q "ultralytics>=8.1.0" torch torchvision opencv-python pyyaml pandas numpy matplotlib seaborn pillow

import sys
import os
import shutil
import json
import hashlib
import time
from datetime import datetime, timezone
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import torch
import ultralytics
from ultralytics import YOLO

print(f"[OK] Python {sys.version.split()[0]} | PyTorch {torch.__version__} | Ultralytics {ultralytics.__version__}")
ultralytics.checks()

In [ ]:
# 2. Google Drive Mount & Directory Hierarchy
try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_base = Path("/content/drive/MyDrive")
except Exception as e:
    print("[WARN] google.colab.drive unavailable; fallback to ./mock_drive simulation.")
    drive_base = Path("./mock_drive")

# Persistent Google Drive Trees
DRIVE_ROOT = drive_base / "ECOSETU_AI"
DATASET_DRIVE_DIR = DRIVE_ROOT / "datasets" / "processed" / "material-detection-v0.1.0"

# Baseline V1 (Strictly preserved untouched)
V1_MODEL_DRIVE_DIR = DRIVE_ROOT / "models" / "material-detection" / "material-detection-v0.1.0"

# Experiment V2 Target Directory
V2_MODEL_DRIVE_DIR = DRIVE_ROOT / "models" / "material-detection" / "material-detection-v0.2.0"
V2_MODEL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Ephemeral Scratch Workspace on Colab NVMe
SCRATCH_ROOT = Path("/content/scratch_training_v2")
SCRATCH_DATASET = SCRATCH_ROOT / "material-detection-v0.1.0"
SCRATCH_RUNS = SCRATCH_ROOT / "runs"
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

print("==================================================")
print(f"Dataset Path:             {DATASET_DRIVE_DIR}")
print(f"V1 Model (Preserved):     {V1_MODEL_DRIVE_DIR}")
print(f"V2 Model Output (New):    {V2_MODEL_DRIVE_DIR}")
print(f"Scratch Root:             {SCRATCH_ROOT}")
print("==================================================")

In [ ]:
# 3. GPU Accelerator Verification
cuda_available = torch.cuda.is_available()
if not cuda_available:
    print("WARNING: CUDA GPU unavailable. Training on CPU will be slow.")
    device_arg = "cpu"
    gpu_name = "CPU (Simulation)"
    cuda_version = "None"
    vram_gb = 0.0
else:
    device_arg = 0
    gpu_name = torch.cuda.get_device_name(0)
    cuda_version = torch.version.cuda
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[OK] GPU: {gpu_name} ({vram_gb:.2f} GB VRAM) | CUDA {cuda_version}")

In [ ]:
# 4. Strict Pre-Training Quality Gate Verification
print("Verifying dataset integrity on Google Drive...")
assert DATASET_DRIVE_DIR.exists(), f"Dataset directory missing: {DATASET_DRIVE_DIR}"

# Check data.yaml classes
yaml_drive_file = DATASET_DRIVE_DIR / "data.yaml"
assert yaml_drive_file.exists(), "data.yaml missing from dataset directory"

with open(yaml_drive_file, "r", encoding="utf-8") as yf:
    data_cfg = yaml.safe_load(yf)

expected_classes = {0: "KEYBOARD_MOUSE", 1: "MOBILE_PHONE", 2: "TABLET"}
class_names = ["KEYBOARD_MOUSE", "MOBILE_PHONE", "TABLET"]
actual_names = data_cfg.get("names", {})
if isinstance(actual_names, list):
    actual_names = {i: n for i, n in enumerate(actual_names)}
assert actual_names == expected_classes, f"Class names mismatch: {actual_names} != {expected_classes}"

# Check manifest status
manifest_drive_file = DATASET_DRIVE_DIR / "dataset_manifest.json"
assert manifest_drive_file.exists(), "dataset_manifest.json missing"
with open(manifest_drive_file, "r", encoding="utf-8") as mf:
    manifest_data = json.load(mf)
assert manifest_data.get("quality_gate_status") == "READY_FOR_TRAINING", f"Quality gate not READY: {manifest_data.get('quality_gate_status')}"

# Verify physical file counts
train_imgs = list((DATASET_DRIVE_DIR / "images" / "train").glob("*.*"))
val_imgs = list((DATASET_DRIVE_DIR / "images" / "val").glob("*.*"))
test_imgs = list((DATASET_DRIVE_DIR / "images" / "test").glob("*.*"))
train_lbls = list((DATASET_DRIVE_DIR / "labels" / "train").glob("*.txt"))
val_lbls = list((DATASET_DRIVE_DIR / "labels" / "val").glob("*.txt"))
test_lbls = list((DATASET_DRIVE_DIR / "labels" / "test").glob("*.txt"))

assert len(train_imgs) == 99 and len(train_lbls) == 99, f"Train count mismatch: {len(train_imgs)} imgs, {len(train_lbls)} lbls"
assert len(val_imgs) == 28 and len(val_lbls) == 28, f"Val count mismatch: {len(val_imgs)} imgs, {len(val_lbls)} lbls"
assert len(test_imgs) == 15 and len(test_lbls) == 15, f"Test count mismatch: {len(test_imgs)} imgs, {len(test_lbls)} lbls"
assert len(train_imgs) + len(val_imgs) + len(test_imgs) == 142, "Total images != 142"

print("==================================================")
print("DATASET INTEGRITY VERIFIED (142 IMAGES / 3 CLASSES)")
print("==================================================")
print(f"Train: {len(train_imgs)} images, {len(train_lbls)} labels")
print(f"Val:   {len(val_imgs)} images, {len(val_lbls)} labels")
print(f"Test:  {len(test_imgs)} images, {len(test_lbls)} labels")
print(f"Quality Gate Status: {manifest_data.get('quality_gate_status')}")
print("==================================================")

In [ ]:
# 5. High-Speed Scratch Staging on Colab NVMe
print("Staging dataset from Google Drive to local NVMe scratch...")
if SCRATCH_DATASET.exists():
    shutil.rmtree(SCRATCH_DATASET)
SCRATCH_DATASET.mkdir(parents=True, exist_ok=True)

for s in ["train", "val", "test"]:
    (SCRATCH_DATASET / "images" / s).mkdir(parents=True, exist_ok=True)
    (SCRATCH_DATASET / "labels" / s).mkdir(parents=True, exist_ok=True)
    for img_f in (DATASET_DRIVE_DIR / "images" / s).glob("*.*"):
        shutil.copy2(img_f, SCRATCH_DATASET / "images" / s / img_f.name)
    for lbl_f in (DATASET_DRIVE_DIR / "labels" / s).glob("*.txt"):
        shutil.copy2(lbl_f, SCRATCH_DATASET / "labels" / s / lbl_f.name)

scratch_yaml_cfg = {
    "path": SCRATCH_DATASET.resolve().as_posix(),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 3,
    "names": expected_classes,
}
scratch_yaml_path = SCRATCH_DATASET / "data.yaml"
with open(scratch_yaml_path, "w", encoding="utf-8") as yf:
    yaml.dump(scratch_yaml_cfg, yf, sort_keys=False)

print(f"[OK] 142 images staged to scratch at: {SCRATCH_DATASET}")

In [ ]:
# 6. Initialize YOLOv8n Architecture & Document Controlled Hyperparameters
print("Initializing base checkpoint: yolov8n.pt ...")
model = YOLO("yolov8n.pt")

# Document Controlled Hyperparameters for V2
v2_hyperparameters = {
    "base_checkpoint": "yolov8n.pt",
    "architecture": "YOLOv8n",
    "imgsz": 640,
    "epochs": 100,
    "patience": 30,
    "lr0": 0.001,          # Conservative learning rate (V1 used 0.01)
    "lrf": 0.01,           # Final learning rate ratio
    "optimizer": "auto",   # Recommended optimizer
    "seed": 42,
    "batch": -1,           # Auto-batch
    "augmentations": {
        "hsv_h": 0.0,      # Zero hue shift (protects wire/PCB casing identity)
        "hsv_s": 0.15,
        "hsv_v": 0.15,
        "degrees": 10.0,
        "translate": 0.1,
        "scale": 0.1,
        "fliplr": 0.5,
        "flipud": 0.0,
        "mosaic": 0.0,     # Disabled mosaic to maintain realistic object scales
        "mixup": 0.0,
        "copy_paste": 0.0,
    }
}

print("==================================================")
print("YOLO V2 CONTROLLED EXPERIMENT CONFIGURATION")
print("==================================================")
for k, v in v2_hyperparameters.items():
    print(f"  {k}: {v}")
print("==================================================")

In [ ]:
# 7. Execute Controlled YOLO V2 Training
training_start_time = datetime.now(timezone.utc)
t0 = time.time()

print("STARTING YOLO V2 TRAINING EXECUTION...")
train_results = model.train(
    data=str(scratch_yaml_path),
    imgsz=640,
    epochs=100,
    patience=30,
    lr0=0.001,
    lrf=0.01,
    batch=-1,
    optimizer="auto",
    seed=42,
    device=device_arg,
    workers=2,
    project=str(SCRATCH_RUNS),
    name="material-detection-v0.2.0",
    exist_ok=True,
    hsv_h=0.0,
    hsv_s=0.15,
    hsv_v=0.15,
    degrees=10.0,
    translate=0.1,
    scale=0.1,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    verbose=True
)

training_duration_seconds = round(time.time() - t0, 2)
print(f"\n[OK] Training completed in {training_duration_seconds} seconds ({training_duration_seconds/60:.2f} mins).")

In [ ]:
# 8. Evaluate on Validation Split & Select Operational Confidence Threshold
run_dir = SCRATCH_RUNS / "material-detection-v0.2.0"
best_pt_v2 = run_dir / "weights" / "best.pt"
assert best_pt_v2.exists(), f"best.pt missing at {best_pt_v2}"

v2_model = YOLO(str(best_pt_v2))

# Standard mAP validation on val partition
val_res = v2_model.val(data=str(scratch_yaml_path), split="val", imgsz=640, device=device_arg)
val_metrics = {
    "precision": round(float(val_res.box.mp), 4) if hasattr(val_res.box, "mp") else None,
    "recall": round(float(val_res.box.mr), 4) if hasattr(val_res.box, "mr") else None,
    "mAP50": round(float(val_res.box.map50), 4) if hasattr(val_res.box, "map50") else None,
    "mAP50_95": round(float(val_res.box.map), 4) if hasattr(val_res.box, "map") else None,
}

# Evaluate Multiple Operational Thresholds on Validation Set ONLY (Tradeoff Table)
val_images = sorted(list((SCRATCH_DATASET / "images" / "val").glob("*.*")))
val_labels = sorted(list((SCRATCH_DATASET / "labels" / "val").glob("*.txt")))

val_gt_boxes = []
for lbl_p in val_labels:
    with open(lbl_p, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                val_gt_boxes.append(int(parts[0]))
total_val_gt = len(val_gt_boxes)

threshold_candidates = [0.10, 0.25, 0.50, 0.75]
val_operating_tradeoff = {}

for th in threshold_candidates:
    tp = 0
    fp = 0
    for img_p in val_images:
        preds = v2_model.predict(source=str(img_p), conf=th, imgsz=640, device=device_arg, verbose=False)
        for r in preds:
            # Approximate TP vs FP on validation
            n_pred = len(r.boxes)
            lbl_p = SCRATCH_DATASET / "labels" / "val" / f"{img_p.stem}.txt"
            n_gt = 0
            if lbl_p.exists():
                with open(lbl_p, "r") as lf: n_gt = len([ln for ln in lf if ln.strip()])
            matched = min(n_pred, n_gt)
            tp += matched
            fp += max(0, n_pred - n_gt)
    fn = max(0, total_val_gt - tp)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    val_operating_tradeoff[str(th)] = {
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1": round(f1, 4),
        "tp": tp, "fp": fp, "fn": fn
    }

# Select operational threshold with highest F1 score on validation set
best_val_th = max(val_operating_tradeoff.keys(), key=lambda k: val_operating_tradeoff[k]["f1"])
selected_operational_conf = float(best_val_th)

print("==================================================")
print("VALIDATION-DRIVEN OPERATIONAL THRESHOLD SELECTION")
print("==================================================")
print(f"Validation mAP50:    {val_metrics['mAP50']}")
print(f"Validation mAP50-95: {val_metrics['mAP50_95']}")
print("\nValidation Operating Points Tradeoff:")
for th, res in val_operating_tradeoff.items():
    print(f"  Conf >= {th:<5}: Precision={res['precision']:.4f} | Recall={res['recall']:.4f} | F1={res['f1']:.4f}")
print("==================================================")
print(f"[DECISION] Selected Operational Threshold based on validation F1: conf = {selected_operational_conf}")
print("==================================================")

In [ ]:
# 9. Held-Out Test Split Evaluation (Evaluated Once at Selected Threshold)
print(f"Evaluating once on held-out test split (15 images) at selected conf={selected_operational_conf}...")
test_res = v2_model.val(data=str(scratch_yaml_path), split="test", imgsz=640, conf=selected_operational_conf, device=device_arg)

test_metrics_operational = {
    "operational_conf": selected_operational_conf,
    "precision": round(float(test_res.box.mp), 4) if hasattr(test_res.box, "mp") else None,
    "recall": round(float(test_res.box.mr), 4) if hasattr(test_res.box, "mr") else None,
    "mAP50": round(float(test_res.box.map50), 4) if hasattr(test_res.box, "map50") else None,
    "mAP50_95": round(float(test_res.box.map), 4) if hasattr(test_res.box, "map") else None,
}

print("==================================================")
print("HELD-OUT TEST RESULTS AT OPERATIONAL THRESHOLD")
print("==================================================")
print(f"Operating Threshold: conf >= {selected_operational_conf}")
print(f"Precision:           {test_metrics_operational['precision']}")
print(f"Recall:              {test_metrics_operational['recall']}")
print(f"mAP50:               {test_metrics_operational['mAP50']}")
print(f"mAP50-95:            {test_metrics_operational['mAP50_95']}")
print("==================================================")

In [ ]:
# 10. Export Artifacts & Model Weights to Google Drive V2 Target
print(f"Exporting V2 artifacts to Google Drive: {V2_MODEL_DRIVE_DIR} ...")
artifacts_to_copy = [
    ("weights/best.pt", "best.pt"),
    ("weights/last.pt", "last.pt"),
    ("args.yaml", "args.yaml"),
    ("results.csv", "results.csv"),
    ("results.png", "results.png"),
    ("confusion_matrix.png", "confusion_matrix.png"),
    ("confusion_matrix_normalized.png", "confusion_matrix_normalized.png"),
    ("PR_curve.png", "PR_curve.png"),
    ("F1_curve.png", "F1_curve.png"),
]

# Create weights subdirectory in drive target
(V2_MODEL_DRIVE_DIR / "weights").mkdir(parents=True, exist_ok=True)
for src_rel, dst_name in artifacts_to_copy:
    src_p = run_dir / src_rel
    dst_p = V2_MODEL_DRIVE_DIR / dst_name
    if src_p.exists():
        shutil.copy2(src_p, dst_p)
        # Also populate weights/ subfolder directly
        if dst_name in ["best.pt", "last.pt"]:
            shutil.copy2(src_p, V2_MODEL_DRIVE_DIR / "weights" / dst_name)
        print(f"  ✔ Saved {dst_name}")

# Compute SHA-256 of persisted V2 best.pt
best_pt_drive_v2 = V2_MODEL_DRIVE_DIR / "weights" / "best.pt"
if not best_pt_drive_v2.exists():
    best_pt_drive_v2 = V2_MODEL_DRIVE_DIR / "best.pt"
hasher = hashlib.sha256()
with open(best_pt_drive_v2, "rb") as bf:
    for chunk in iter(lambda: bf.read(65536), b""):
        hasher.update(chunk)
v2_sha256 = hasher.hexdigest()

# Determine actual epochs trained
actual_epochs = None
results_csv_path = run_dir / "results.csv"
if results_csv_path.exists():
    try:
        df_res = pd.read_csv(results_csv_path)
        actual_epochs = int(len(df_res))
    except Exception:
        pass
if actual_epochs is None and hasattr(train_results, "epoch"):
    actual_epochs = len(train_results.epoch)

# Save training_metadata.json
training_metadata = {
    "dataset_version": "material-detection-v0.1.0",
    "model_architecture": "YOLOv8n",
    "ultralytics_version": ultralytics.__version__,
    "python_version": sys.version.split()[0],
    "pytorch_version": torch.__version__,
    "cuda_version": cuda_version,
    "gpu": gpu_name,
    "epochs_requested": 100,
    "actual_epochs": actual_epochs,
    "learning_rate": 0.001,
    "learning_rate_initial": 0.001,
    "learning_rate_final_ratio": 0.01,
    "optimizer": "auto",
    "seed": 42,
    "image_size": 640,
    "batch_size": "auto",
    "sha256": v2_sha256,
    "training_date": training_start_time.isoformat(),
    "training_duration_seconds": training_duration_seconds,
}
with open(V2_MODEL_DRIVE_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=2)

print(f"[OK] Saved training_metadata.json (SHA-256: {v2_sha256})")

In [ ]:
# 11. Generate Diagnostic Predictions on All 15 Held-Out Test Images
diag_pred_dir = V2_MODEL_DRIVE_DIR / "diagnostic_visuals"
diag_pred_dir.mkdir(parents=True, exist_ok=True)

test_images = sorted(list((SCRATCH_DATASET / "images" / "test").glob("*.*")))
test_labels_dir = SCRATCH_DATASET / "labels" / "test"

print(f"Generating visual comparison predictions for all {len(test_images)} test images...")
visual_records = []

for img_p in test_images:
    preds = v2_model.predict(source=str(img_p), conf=selected_operational_conf, imgsz=640, device=device_arg, verbose=False)
    for res in preds:
        save_file = diag_pred_dir / f"pred_{img_p.name}"
        annotated_bgr = res.plot()
        im = Image.fromarray(annotated_bgr[..., ::-1])
        im.save(save_file)
        
        boxes_info = []
        for b in res.boxes:
            cid = int(b.cls[0].item())
            conf = float(b.conf[0].item())
            xywhn = b.xywhn[0].tolist()
            boxes_info.append({
                "class_id": cid,
                "class_name": class_names[cid] if cid < len(class_names) else "UNKNOWN",
                "confidence": round(conf, 4),
                "bbox_xywhn": [round(v, 4) for v in xywhn]
            })
        visual_records.append({
            "image": img_p.name,
            "detections": boxes_info,
            "visual_file": str(save_file)
        })

print(f"[OK] Saved {len(visual_records)} visual prediction images to: {diag_pred_dir}")

In [ ]:
# 12. Generate v1_vs_v2_comparison.json and v1_vs_v2_comparison.md
comparison_json_path = V2_MODEL_DRIVE_DIR / "v1_vs_v2_comparison.json"
comparison_md_path = V2_MODEL_DRIVE_DIR / "v1_vs_v2_comparison.md"

v1_val_map50 = 0.2386
v1_val_map50_95 = 0.1725
v1_test_map50 = 0.1709
v1_test_map50_95 = 0.0950
v1_test_p_at_025 = 0.4118
v1_test_r_at_025 = 0.9333

v2_val_map50 = val_metrics.get("mAP50", 0.0)
v2_val_map50_95 = val_metrics.get("mAP50_95", 0.0)
v2_test_map50 = test_metrics_operational.get("mAP50", 0.0)
v2_test_map50_95 = test_metrics_operational.get("mAP50_95", 0.0)
v2_test_p = test_metrics_operational.get("precision", 0.0)
v2_test_r = test_metrics_operational.get("recall", 0.0)

comparison_data = {
    "comparison_date": datetime.now(timezone.utc).isoformat(),
    "production_status": "BASELINE V2 — EVALUATION REQUIRED",
    "dataset": "material-detection-v0.1.0 (142 authentic images)",
    "architecture": "YOLOv8n",
    "hyperparameter_deltas": {
        "V1": {"lr0": 0.01, "lrf": 0.01, "patience": 20, "epochs": 100},
        "V2": {"lr0": 0.001, "lrf": 0.01, "patience": 30, "epochs": 100}
    },
    "validation_comparison": {
        "mAP50": {"V1": v1_val_map50, "V2": v2_val_map50, "delta": round(v2_val_map50 - v1_val_map50, 4)},
        "mAP50_95": {"V1": v1_val_map50_95, "V2": v2_val_map50_95, "delta": round(v2_val_map50_95 - v1_val_map50_95, 4)},
    },
    "held_out_test_comparison": {
        "mAP50": {"V1": v1_test_map50, "V2": v2_test_map50, "delta": round(v2_test_map50 - v1_test_map50, 4)},
        "mAP50_95": {"V1": v1_test_map50_95, "V2": v2_test_map50_95, "delta": round(v2_test_map50_95 - v1_test_map50_95, 4)},
        "precision": {"V1_at_025": v1_test_p_at_025, "V2_operational": v2_test_p},
        "recall": {"V1_at_025": v1_test_r_at_025, "V2_operational": v2_test_r},
    },
    "operational_threshold_selected": selected_operational_conf,
    "validation_tradeoff": val_operating_tradeoff,
}

with open(comparison_json_path, "w", encoding="utf-8") as f:
    json.dump(comparison_data, f, indent=2)

md_report = f"""# ECOSETU YOLO V1 VS V2 CONTROLLED EXPERIMENT COMPARISON

**Status:** `BASELINE V2 — EVALUATION REQUIRED`  
**Generated At:** `{comparison_data["comparison_date"]}`  
**Dataset:** `material-detection-v0.1.0` (142 authentic images)  

---

## 1. Experimental Setup & Controlled Hypotheses

| Parameter | Baseline V1 | Experiment V2 | Controlled Objective |
| :--- | :--- | :--- | :--- |
| **Architecture** | YOLOv8n | YOLOv8n | Controlled comparison |
| **Dataset** | 142 authentic images | 142 authentic images | Zero dataset modification |
| **Initial LR (`lr0`)** | `0.01` | `0.001` | Reduce gradient destabilization on small dataset |
| **Patience** | `20` | `30` | Prevent premature early stopping |
| **Selected Conf Threshold** | `0.25` | `{selected_operational_conf}` | Validation-derived optimal F1 point |

---

## 2. Validation & Held-Out Test Metric Comparison

### Validation Partition (28 Images):
| Metric | Baseline V1 | Experiment V2 | Delta |
| :--- | :---: | :---: | :---: |
| **mAP50** | `{v1_val_map50}` | `{v2_val_map50}` | `{v2_val_map50 - v1_val_map50:+.4f}` |
| **mAP50-95** | `{v1_val_map50_95}` | `{v2_val_map50_95}` | `{v2_val_map50_95 - v1_val_map50_95:+.4f}` |

### Held-Out Test Partition (15 Images):
| Metric | Baseline V1 | Experiment V2 | Delta |
| :--- | :---: | :---: | :---: |
| **mAP50** | `{v1_test_map50}` | `{v2_test_map50}` | `{v2_test_map50 - v1_test_map50:+.4f}` |
| **mAP50-95** | `{v1_test_map50_95}` | `{v2_test_map50_95}` | `{v2_test_map50_95 - v1_test_map50_95:+.4f}` |
| **Operational Precision** | `{v1_test_p_at_025}` | `{v2_test_p}` | `{v2_test_p - v1_test_p_at_025:+.4f}` |
| **Operational Recall** | `{v1_test_r_at_025}` | `{v2_test_r}` | `{v2_test_r - v1_test_r_at_025:+.4f}` |

---

## 3. Production Readiness Determination

- **Determination:** `BASELINE V2 — EVALUATION REQUIRED`
- **Rationale:** Model is an experimental improvement baseline. Edge deployment to mobile app remains blocked until field validation on diverse background collections.
"""

with open(comparison_md_path, "w", encoding="utf-8") as f:
    f.write(md_report)

print("==================================================")
print(f"[OK] Comparison JSON saved to: {comparison_json_path}")
print(f"[OK] Comparison MD saved to:   {comparison_md_path}")
print("==================================================")

In [ ]:
# 13. Final Experiment Summary
print("==================================================")
print("ECOSETU YOLO V2 TRAINING EXPERIMENT COMPLETE")
print("==================================================")
print("Status:       BASELINE V2 — EVALUATION REQUIRED")
print(f"Model Dir:    {V2_MODEL_DRIVE_DIR}")
print(f"V1 Preserved: {V1_MODEL_DRIVE_DIR}")
print("")
print(f"Validation mAP50:    {val_metrics.get('mAP50')}")
print(f"Held-Out Test mAP50: {test_metrics_operational.get('mAP50')}")
print(f"Selected Threshold:  conf >= {selected_operational_conf}")
print("==================================================")